# 03 · App setup (run once, after creating the app)
1. Creates the `preferences` table (per-user settings from the app's ⚙️ Settings tab)
2. Grants the app's service principal access to the `career_agent` schema

Before running: **Compute → Apps → Create app → Custom** named `career-agent`
(see README → *Databricks App*).

In [ ]:
%pip install -q --upgrade "databricks-sdk>=0.40" pyyaml

In [ ]:
dbutils.library.restartPython()

In [ ]:
import os, sys
sys.path.append(os.path.abspath("../src"))
from career_agent import config

dbutils.widgets.text("app_name", "career-agent")
app_name = dbutils.widgets.get("app_name")

## 1. Preferences table

In [ ]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {config.table('preferences')} (
  user_email STRING NOT NULL,
  prefs      STRING COMMENT 'JSON: target_roles, level, remote, cities, min_salary_usd, stop_words, excluded_industries, telegram_chat_id, llm_provider, llm_model',
  updated_at TIMESTAMP
) COMMENT 'Per-user job-search settings written by the Career Agent app'
""")
display(spark.sql(f"DESCRIBE TABLE {config.table('preferences')}"))

## 2. Grant the app's service principal access

In [ ]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
app = w.apps.get(app_name)
sp = app.service_principal_client_id
print("App URL:", app.url)
print("Service principal:", sp)

for stmt in [
    f"GRANT USE CATALOG ON CATALOG {config.CATALOG} TO `{sp}`",
    f"GRANT USE SCHEMA, SELECT, MODIFY ON SCHEMA {config.CATALOG}.{config.SCHEMA} TO `{sp}`",
]:
    spark.sql(stmt)
    print("✅", stmt)